# Deep Past Initiative: DINO EMA Pretraining & Evaluation (Single GPU)
Full pipeline for Dual-Objective Sequence-Level KD DINO EMA Pretraining on a single GPU, built for direct execution on Kaggle.

## 1. Install Dependencies

In [ ]:
!pip install -q transformers sacrebleu kagglehub


## 2. Write Training Script
Write the training logic to `train_script.py` and run it via the OS shell to avoid Jupyter's forked-process CUDA limitations.

In [ ]:
%%writefile train_script.py
#!/usr/bin/env python3
"""
Phase 1: DINO EMA Pretraining + Sequence-Level KD for ByT5 Akkadian (Single GPU)
=================================================================================

Architecture (Length-Preserving + Asymmetric KD):
  - Length-preserving corruption: ~15% of bytes → random bytes (L = L')
  - Teacher (EMA, eval): clean input → encoder → proj head → p_T (Encoder)
                         clean input → decoder -> p_T_dec (Decoder)
  - Student (trainable): corrupted input → encoder → proj head → p_S (Encoder)
                         corrupted input → decoder -> p_S_dec (Decoder)
  - Token-wise DINO loss: -mean(sum(p_T · log(p_S))) (Encoder self-distillation)
  - Sequence-Level KD: KL-Divergence(p_T_dec || p_S_dec) (Decoder knowledge distillation)
  - Teacher is updated strictly via Exponential Moving Average (EMA).
  - L_total = λ_DINO * L_DINO + λ_KD * L_KD
"""

import os
import gc
import re
import math
import copy
import time
import random
import logging
import argparse
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm


@dataclass
class DINOEMAConfig:
    # Paths
    model_path: str = "/kaggle/input/byt5-akkadian-optimized-34x/byt5-akkadian-optimized-34x"
    data_path: str = "/kaggle/input/deep-past-initiative-machine-translation/published_texts.csv"
    output_dir: str = "/kaggle/working/dino_ema_output"

    # Model dimensions
    d_model: int = 1536
    proj_hidden: int = 3072
    proj_output: int = 256

    # Training
    batch_size: int = 4
    grad_accum: int = 8  # effective batch size = 32
    epochs: int = 10
    lr: float = 1e-4
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    warmup_ratio: float = 0.05

    # Loss weights
    lambda_dino: float = 1.0
    lambda_kd: float = 1.0

    # EMA / DINO
    ema_base: float = 0.996
    tau_s: float = 0.1
    tau_t_start: float = 0.04
    tau_t_end: float = 0.07
    center_momentum: float = 0.9

    # Length-preserving corruption
    mask_ratio: float = 0.15

    # Tokenizer
    max_length: int = 512
    pad_token_id: int = 0
    eos_token_id: int = 1

    # Translation sample check (labeled data)
    train_data_path: str = ""  # train.csv with transliteration/translation columns
    sample_check_every: int = 80  # steps between translation sample checks

    # Checkpointing
    save_every_steps: int = 500
    log_every_steps: int = 10
    seed: int = 42

    # Device / precision
    device: str = "cuda"
    use_bf16: bool = True
    gradient_checkpointing: bool = True

    def __post_init__(self):
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)


def parse_args() -> DINOEMAConfig:
    parser = argparse.ArgumentParser(description="DINO EMA + Seq-KD for ByT5 Akkadian (Single GPU)")
    cfg = DINOEMAConfig()
    for k, v in vars(cfg).items():
        t = type(v) if v is not None else str
        if t == bool:
            parser.add_argument(f"--{k}", type=lambda x: x.lower() in ("true", "1"), default=v)
        else:
            parser.add_argument(f"--{k}", type=t, default=v)
    args = parser.parse_args()
    return DINOEMAConfig(**vars(args))


def setup_logging(output_dir: str) -> logging.Logger:
    Path(output_dir).mkdir(exist_ok=True, parents=True)
    for h in logging.root.handlers[:]:
        logging.root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(levelname)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(output_dir) / "dino_ema_train.log"),
        ],
    )
    return logging.getLogger("dino_ema_train")


def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# ──────────────────────────────────────────────────────────────
# Preprocessing
# ──────────────────────────────────────────────────────────────

_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a": "á", "e": "é", "i": "í", "u": "ú",
                         "A": "Á", "E": "É", "I": "Í", "U": "Ú"})
_GRAVE = str.maketrans({"a": "à", "e": "è", "i": "ì", "u": "ù",
                         "A": "À", "E": "È", "I": "Ì", "U": "Ù"})


def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


_ALLOWED_FRACS = [
    (1/6, "0.16666"), (1/4, "0.25"), (1/3, "0.33333"),
    (1/2, "0.5"), (2/3, "0.66666"), (3/4, "0.75"), (5/6, "0.83333"),
]
_FRAC_TOL = 2e-3
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")


def _canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= _FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")


_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)


def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ": "h", "Ḫ": "H", "ʾ": "",
    "₀": "0", "₁": "1", "₂": "2", "₃": "3", "₄": "4",
    "₅": "5", "₆": "6", "₇": "7", "₈": "8", "₉": "9",
    "—": "-", "–": "-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")
_WS_RE = re.compile(r"\s+")

_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {
    "0.8333": "⅚", "0.6666": "⅔", "0.3333": "⅓", "0.1666": "⅙",
    "0.625": "⅝", "0.75": "¾", "0.25": "¼", "0.5": "½",
}


def _frac_repl(m: re.Match) -> str:
    return _EXACT_FRAC_MAP[m.group(0)]


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)
        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        ser = ser.str.replace(_FLOAT_RE,
                              lambda m: _canon_decimal(float(m.group(1))), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()


# ──────────────────────────────────────────────────────────────
# Dataset + DataLoader
# ──────────────────────────────────────────────────────────────

class DINOAkkadianDataset(Dataset):
    def __init__(self, csv_path: str, tokenizer, max_length: int, logger: logging.Logger):
        df = pd.read_csv(csv_path, encoding="utf-8")
        raw_texts = df["transliteration"].tolist()
        logger.info(f"Loaded {len(raw_texts)} texts from {csv_path}")

        preprocessor = OptimizedPreprocessor()
        texts = preprocessor.preprocess_batch(raw_texts)
        texts = [t for t in texts if t and t.strip()]
        logger.info(f"After preprocessing/filtering: {len(texts)} texts")

        # [Fix #2] Pre-tokenize once at init — avoids re-tokenizing every __getitem__ call
        logger.info("Pre-tokenizing all texts...")
        self.input_ids: List[torch.Tensor] = []
        for text in texts:
            enc = tokenizer(
                text,
                max_length=max_length,
                truncation=True,
                return_tensors="pt",
                add_special_tokens=True,
            )
            self.input_ids.append(enc.input_ids.squeeze(0))
        logger.info(f"Pre-tokenization complete: {len(self.input_ids)} samples cached")

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx]


def collate_fn(batch: List[torch.Tensor], pad_token_id: int = 0) -> torch.Tensor:
    max_len = max(x.size(0) for x in batch)
    padded = torch.full((len(batch), max_len), pad_token_id, dtype=torch.long)
    for i, x in enumerate(batch):
        padded[i, :x.size(0)] = x
    return padded


# ──────────────────────────────────────────────────────────────
# Length-Preserving Corruption
# ──────────────────────────────────────────────────────────────

def length_preserving_corruption(
    input_ids: torch.Tensor,
    mask_ratio: float = 0.15,
    pad_token_id: int = 0,
    eos_token_id: int = 1,
) -> Tuple[torch.Tensor, torch.Tensor]:
    # [Fix #4] Fully vectorized — no Python for-loop over batch items
    B, L = input_ids.shape
    device = input_ids.device

    content_mask = (input_ids != pad_token_id) & (input_ids != eos_token_id)  # [B, L]

    # Random scores: content positions get uniform [0,1), non-content gets 2.0 (pushed to back)
    rand_scores = torch.rand(B, L, device=device)
    rand_scores[~content_mask] = 2.0

    # rank[b, l] = position of token l in ascending score order
    rank = rand_scores.argsort(dim=1).argsort(dim=1)

    n_content = content_mask.sum(dim=1)  # [B]
    n_corrupt = (n_content.float() * mask_ratio).round().clamp(min=1).long()  # [B]

    # Positions with rank < n_corrupt[b] AND content → corrupted
    corruption_mask = (rank < n_corrupt.unsqueeze(1)) & content_mask  # [B, L]

    random_bytes = torch.randint(3, 259, (B, L), device=device)
    corrupted = input_ids.clone()
    corrupted[corruption_mask] = random_bytes[corruption_mask]

    return corrupted, corruption_mask


# ──────────────────────────────────────────────────────────────
# Projection Head + DINO Wrapper
# ──────────────────────────────────────────────────────────────

class DINOProjectionHead(nn.Module):
    def __init__(self, d_model: int = 1536, hidden: int = 3072, output: int = 256,
                 initial_g: float = 10.0):
        super().__init__()
        self.linear1 = nn.Linear(d_model, hidden)
        self.act = nn.GELU()
        self.ln = nn.LayerNorm(hidden)
        self.last_layer = nn.utils.weight_norm(nn.Linear(hidden, output, bias=False))
        self.last_layer.weight_g.data.fill_(initial_g)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = self.act(x)
        x = self.ln(x)
        x = F.normalize(x, dim=-1)
        x = self.last_layer(x)
        return x


class DINOEMA(nn.Module):
    def __init__(self, cfg: DINOEMAConfig, logger: logging.Logger):
        super().__init__()
        self.cfg = cfg
        self.logger = logger
        self._current_tau_t = cfg.tau_t_start

        logger.info(f"Loading student model from {cfg.model_path}")
        self.student = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_path)

        if cfg.gradient_checkpointing:
            self.student.gradient_checkpointing_enable()
            logger.info("Gradient checkpointing enabled")

        logger.info("Creating teacher (deep copy of student)")
        self.teacher = copy.deepcopy(self.student)
        for p in self.teacher.parameters():
            p.requires_grad = False

        self.student_head = DINOProjectionHead(cfg.d_model, cfg.proj_hidden, cfg.proj_output)
        self.teacher_head = DINOProjectionHead(cfg.d_model, cfg.proj_hidden, cfg.proj_output)
        self.teacher_head.load_state_dict(self.student_head.state_dict())
        for p in self.teacher_head.parameters():
            p.requires_grad = False

        # Teacher는 항상 eval — forward 안에서 매번 세팅할 필요 없음
        self.teacher.eval()
        self.teacher_head.eval()

        self.register_buffer("center", torch.zeros(cfg.proj_output))

        n_student = sum(p.numel() for p in self.student.parameters() if p.requires_grad)
        n_head = sum(p.numel() for p in self.student_head.parameters() if p.requires_grad)
        logger.info(f"Trainable Student params: {n_student:,} + Trainable head params: {n_head:,}")

    def train(self, mode: bool = True):
        """model.train() 호출 시 teacher는 항상 eval로 고정."""
        super().train(mode)
        self.teacher.eval()
        self.teacher_head.eval()
        return self

    @torch.no_grad()
    def update_teacher(self, momentum: float):
        for t_p, s_p in zip(self.teacher.parameters(), self.student.parameters()):
            t_p.data.mul_(momentum).add_(s_p.data, alpha=1.0 - momentum)
        for t_p, s_p in zip(self.teacher_head.parameters(), self.student_head.parameters()):
            t_p.data.mul_(momentum).add_(s_p.data, alpha=1.0 - momentum)

    def forward(self, input_ids: torch.Tensor) -> dict:
        cfg = self.cfg

        corrupted_ids, corruption_mask = length_preserving_corruption(
            input_ids,
            mask_ratio=cfg.mask_ratio,
            pad_token_id=cfg.pad_token_id,
            eos_token_id=cfg.eos_token_id,
        )

        attn_mask = (input_ids != cfg.pad_token_id).long()
        valid_mask = attn_mask.bool()

        labels = input_ids.clone()
        labels[~valid_mask] = -100

        # [Fix #3] Pass decoder_input_ids directly instead of labels
        # — avoids HuggingFace computing an internal CE loss we never use
        decoder_input_ids = self.student._shift_right(labels)

        # ── Teacher (clean input, no gradient) ──
        with torch.no_grad():
            teacher_outputs = self.teacher(
                input_ids=input_ids,
                attention_mask=attn_mask,
                decoder_input_ids=decoder_input_ids,
            )
            t_enc_out = teacher_outputs.encoder_last_hidden_state
            z_T = self.teacher_head(t_enc_out)

            z_T_centered = z_T - self.center.unsqueeze(0).unsqueeze(0)
            p_T = F.softmax(z_T_centered / self._current_tau_t, dim=-1)

            if valid_mask.any():
                batch_center = z_T[valid_mask].mean(dim=0)
                self.center.mul_(cfg.center_momentum).add_(batch_center, alpha=1.0 - cfg.center_momentum)

            t_logits = teacher_outputs.logits

        # ── Student (corrupted input, track gradients) ──
        student_outputs = self.student(
            input_ids=corrupted_ids,
            attention_mask=attn_mask,
            decoder_input_ids=decoder_input_ids,
        )

        s_enc_out = student_outputs.encoder_last_hidden_state
        z_S = self.student_head(s_enc_out)
        p_S_log = F.log_softmax(z_S / cfg.tau_s, dim=-1)

        s_logits = student_outputs.logits

        # ── Loss computation ──
        per_token_dino_loss = -(p_T * p_S_log).sum(dim=-1)
        dino_loss = per_token_dino_loss[valid_mask].mean() if valid_mask.any() else per_token_dino_loss.mean()

        t_probs = F.softmax(t_logits, dim=-1)
        s_log_probs = F.log_softmax(s_logits, dim=-1)
        per_token_kd_loss = F.kl_div(s_log_probs, t_probs, reduction='none').sum(dim=-1)
        target_valid_mask = (labels != -100)
        kd_loss = per_token_kd_loss[target_valid_mask].mean() if target_valid_mask.any() else per_token_kd_loss.mean()

        total_loss = cfg.lambda_dino * dino_loss + cfg.lambda_kd * kd_loss

        # ── Diagnostics ──
        with torch.no_grad():
            ln_K = math.log(cfg.proj_output)
            p_S_probs = F.softmax(z_S / cfg.tau_s, dim=-1)
            H_T = -(p_T * (p_T + 1e-10).log()).sum(dim=-1)
            H_S = -(p_S_probs * (p_S_probs + 1e-10).log()).sum(dim=-1)

            if valid_mask.any():
                H_norm_T = (H_T[valid_mask].mean() / ln_K).item()
                H_norm_S = (H_S[valid_mask].mean() / ln_K).item()
                std_z_T = z_T[valid_mask].std().item()
                std_z_S = z_S[valid_mask].std().item()
            else:
                H_norm_T = (H_T.mean() / ln_K).item()
                H_norm_S = (H_S.mean() / ln_K).item()
                std_z_T = z_T.std().item()
                std_z_S = z_S.std().item()

        return {
            "total_loss": total_loss,
            "dino_loss": dino_loss.detach(),
            "kd_loss": kd_loss.detach(),
            "diagnostics": {
                "H_norm_T": H_norm_T,
                "H_norm_S": H_norm_S,
                "std_z_T": std_z_T,
                "std_z_S": std_z_S,
                "corrupt_ratio": corruption_mask.sum().item() / max(valid_mask.sum().item(), 1),
            },
        }


# ──────────────────────────────────────────────────────────────
# Scheduler helpers
# ──────────────────────────────────────────────────────────────

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps, num_training_steps):
    def lr_lambda(current_step):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(
            max(1, num_training_steps - num_warmup_steps)
        )
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


def get_ema_momentum(step: int, total_steps: int, base: float = 0.996) -> float:
    return 1.0 - (1.0 - base) * (math.cos(math.pi * step / total_steps) + 1.0) / 2.0


def get_teacher_temp(step: int, total_steps: int, t_start: float, t_end: float) -> float:
    warmup_steps = int(0.3 * total_steps)
    if step < warmup_steps:
        return t_start + (t_end - t_start) * step / warmup_steps
    return t_end


def gpu_mem_info() -> str:
    if not torch.cuda.is_available():
        return "CPU"
    used = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    return f"{used:.1f}/{total:.0f}GB"


def load_translation_samples(
    train_data_path: str, logger: logging.Logger
) -> List[Tuple[str, str]]:
    """train.csv에서 (전처리된_transliteration, translation) 쌍을 로드."""
    if not train_data_path or not os.path.exists(train_data_path):
        logger.warning(f"train_data_path not found: '{train_data_path}' — sample check disabled")
        return []
    df = pd.read_csv(train_data_path, encoding="utf-8")
    preprocessor = OptimizedPreprocessor()
    srcs = preprocessor.preprocess_batch(df["transliteration"].tolist())
    refs = df["translation"].tolist()
    samples = [
        (s, str(r)) for s, r in zip(srcs, refs)
        if s and str(r) not in ("nan", "", "None")
    ]
    logger.info(f"Loaded {len(samples)} labeled samples for translation checks")
    return samples


@torch.no_grad()
def sample_translation_check(
    model: "DINOEMA",
    tokenizer,
    samples: List[Tuple[str, str]],
    device: torch.device,
    amp_dtype: torch.dtype,
    use_amp: bool,
    logger: logging.Logger,
    global_step: int,
):
    """랜덤 샘플 하나를 뽑아 student 모델의 현재 번역 출력 vs 정답을 로그."""
    src, ref = random.choice(samples)
    input_text = "translate Akkadian to English: " + src

    enc = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(device)

    was_training = model.student.training
    model.student.eval()

    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
        output_ids = model.student.generate(
            input_ids=enc.input_ids,
            attention_mask=enc.attention_mask,
            max_new_tokens=128,
            num_beams=2,
            early_stopping=True,
        )

    if was_training:
        model.student.train()

    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    logger.info(
        f"\n{'─'*60}\n"
        f"[Step {global_step}] Translation Sample Check\n"
        f"  SRC : {src[:150]}\n"
        f"  REF : {ref[:150]}\n"
        f"  PRED: {pred[:150]}\n"
        f"{'─'*60}"
    )


# ──────────────────────────────────────────────────────────────
# Training Loop
# ──────────────────────────────────────────────────────────────

def train(cfg: DINOEMAConfig):
    logger = setup_logging(cfg.output_dir)
    logger.info("=" * 60)
    logger.info("DINO EMA Pretraining + Sequence KD For ByT5 Akkadian (Single GPU)")
    logger.info("=" * 60)
    logger.info(f"  Model  : {cfg.model_path}")
    logger.info(f"  Data   : {cfg.data_path}")
    logger.info(f"  Batch  : {cfg.batch_size} x accum {cfg.grad_accum} = {cfg.batch_size * cfg.grad_accum} effective")
    logger.info(f"  LR     : {cfg.lr}")
    logger.info(f"  bf16   : {cfg.use_bf16}")

    set_seed(cfg.seed)

    device = torch.device(cfg.device if torch.cuda.is_available() else "cpu")
    logger.info(f"Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_path)
    dataset = DINOAkkadianDataset(cfg.data_path, tokenizer, cfg.max_length, logger)

    # 레이블 데이터 로드 (번역 샘플 체크용)
    # Kaggle 경로: published_texts.csv와 같은 디렉토리에 train.csv 존재
    # 예: /kaggle/input/deep-past-initiative-machine-translation/train.csv
    #     /kaggle/input/competitions/deep-past-initiative-machine-translation/train.csv
    train_data_path = cfg.train_data_path
    if not train_data_path:
        # data_path(published_texts.csv)의 디렉토리에서 train.csv를 자동 탐색
        candidate = str(Path(cfg.data_path).parent / "train.csv")
        if os.path.exists(candidate):
            train_data_path = candidate
    translation_samples = load_translation_samples(train_data_path, logger)

    dataloader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=lambda batch: collate_fn(batch, cfg.pad_token_id),
        drop_last=True,
        num_workers=2,
        pin_memory=True,
    )

    model = DINOEMA(cfg, logger).to(device)

    optimizer = torch.optim.AdamW(
        [
            {"params": model.student.parameters()},
            {"params": model.student_head.parameters()},
        ],
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
        betas=(0.9, 0.999),
    )

    batches_per_epoch = len(dataloader)
    total_steps = (batches_per_epoch // cfg.grad_accum) * cfg.epochs
    warmup_steps = int(cfg.warmup_ratio * total_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    # Mixed precision scaler (bf16 doesn't need GradScaler, but fp16 does)
    use_amp = cfg.use_bf16 and device.type == "cuda"
    amp_dtype = torch.bfloat16 if cfg.use_bf16 else torch.float16
    scaler = torch.cuda.amp.GradScaler(enabled=(not cfg.use_bf16 and use_amp))

    logger.info(f"Total optimizer steps: {total_steps} | Warmup: {warmup_steps}")

    # [Fix #5] Cache parameter list for grad clipping — avoids rebuilding list every accum step
    clip_params = list(model.student.parameters()) + list(model.student_head.parameters())

    global_step = 0
    running_dino = 0.0
    running_kd = 0.0
    running_count = 0

    for epoch in range(cfg.epochs):
        model.train()
        optimizer.zero_grad()

        pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{cfg.epochs}", dynamic_ncols=True)

        for batch_idx, input_ids in enumerate(pbar):
            input_ids = input_ids.to(device)

            # Update teacher temperature
            model._current_tau_t = get_teacher_temp(
                global_step, total_steps, cfg.tau_t_start, cfg.tau_t_end
            )

            # Forward pass (with optional autocast)
            with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=use_amp):
                losses = model(input_ids)
                # Scale loss for gradient accumulation
                loss = losses["total_loss"] / cfg.grad_accum

            scaler.scale(loss).backward()

            is_accum_step = (batch_idx + 1) % cfg.grad_accum == 0

            if is_accum_step:
                scaler.unscale_(optimizer)
                # [Fix #5] Use cached param list
                torch.nn.utils.clip_grad_norm_(clip_params, cfg.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                scheduler.step()

                # EMA teacher update
                ema_m = get_ema_momentum(global_step, total_steps, cfg.ema_base)
                model.update_teacher(ema_m)
                # [Fix #1] empty_cache() 제거 — CUDA sync stall 원인

                global_step += 1

            # 번역 샘플 체크 — batch_idx 기준 (tqdm에 보이는 숫자와 일치)
            if translation_samples and (batch_idx + 1) % cfg.sample_check_every == 0:
                sample_translation_check(
                    model, tokenizer, translation_samples,
                    device, amp_dtype, use_amp, logger, global_step,
                )

                if global_step % cfg.log_every_steps == 0:
                    logger.info(
                        f"[Step {global_step}/{total_steps}] "
                        f"DINO={smooth_dino:.4f} KD={smooth_kd:.4f} "
                        f"Total={losses['total_loss'].item():.4f} | "
                        f"EMA={ema_m:.5f} | lr={scheduler.get_last_lr()[0]:.2e} | "
                        f"GPU={gpu_mem_info()}"
                    )

            # Running loss tracking (exponential moving average style)
            running_dino += losses["dino_loss"].item()
            running_kd += losses["kd_loss"].item()
            running_count += 1
            if running_count > 50:
                running_dino -= running_dino / running_count
                running_kd -= running_kd / running_count
                running_count -= 1

            smooth_dino = running_dino / max(1, running_count)
            smooth_kd = running_kd / max(1, running_count)
            ema_m_display = get_ema_momentum(global_step, total_steps, cfg.ema_base)
            pbar.set_postfix_str(
                f"DINO={smooth_dino:.3f} KD={smooth_kd:.3f} EMA={ema_m_display:.4f} GPU={gpu_mem_info()}"
            )

        # ── Epoch checkpoint ──
        epoch_dir = Path(cfg.output_dir) / f"epoch-{epoch + 1}"
        student_dir = epoch_dir / "student"
        teacher_dir = epoch_dir / "teacher"
        student_dir.mkdir(exist_ok=True, parents=True)
        teacher_dir.mkdir(exist_ok=True, parents=True)

        # Student와 Teacher를 분리 저장 — 재학습 시 EMA divergence 보존
        model.student.save_pretrained(student_dir, safe_serialization=False)
        model.teacher.save_pretrained(teacher_dir, safe_serialization=False)
        tokenizer.save_pretrained(epoch_dir)
        torch.save({
            "student_head": model.student_head.state_dict(),
            "teacher_head": model.teacher_head.state_dict(),
            "center": model.center,
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "global_step": global_step,
            "epoch": epoch,
        }, epoch_dir / "dino_state.pt")
        logger.info(f"Epoch {epoch + 1} checkpoint saved → {epoch_dir}")

    # ── Final save ──
    final_dir = Path(cfg.output_dir) / "final"
    student_dir = final_dir / "student"
    teacher_dir = final_dir / "teacher"
    student_dir.mkdir(exist_ok=True, parents=True)
    teacher_dir.mkdir(exist_ok=True, parents=True)
    model.student.save_pretrained(student_dir, safe_serialization=False)
    model.teacher.save_pretrained(teacher_dir, safe_serialization=False)
    tokenizer.save_pretrained(final_dir)
    logger.info("Training complete. Final model saved.")


if __name__ == "__main__":
    cfg = parse_args()
    train(cfg)


## 3. Detect Paths & Download Model

In [ ]:
import os
import kagglehub

# Find competition dataset directory
BASE_DATA_DIR = "/kaggle/input/deep-past-initiative-machine-translation"
for root, dirs, files in os.walk("/kaggle/input"):
    if "published_texts.csv" in files:
        BASE_DATA_DIR = root
        break

# Download baseline model via kagglehub
print("Downloading baseline model via kagglehub...")
dataset_path = kagglehub.dataset_download("assiaben/final-byt5")
print(f"Downloaded to: {dataset_path}")

# Locate model folder (contains config.json)
MODEL_PATH = ""
for root, dirs, files in os.walk(dataset_path):
    if "config.json" in files:
        MODEL_PATH = root
        break

if not MODEL_PATH:
    raise FileNotFoundError(f"config.json not found inside {dataset_path}")

OUTPUT_DIR = "/kaggle/working/dino_ema_output"
PUBLISHED_CSV = os.path.join(BASE_DATA_DIR, "published_texts.csv")
TRAIN_CSV = os.path.join(BASE_DATA_DIR, "train.csv")

os.environ["MODEL_PATH"] = MODEL_PATH
os.environ["PUBLISHED_CSV"] = PUBLISHED_CSV
os.environ["OUTPUT_DIR"] = OUTPUT_DIR
os.environ["TRAIN_CSV"] = TRAIN_CSV

print(f"BASE_DATA_DIR : {BASE_DATA_DIR}")
print(f"MODEL_PATH    : {MODEL_PATH}")
print(f"PUBLISHED_CSV : {PUBLISHED_CSV}")
print(f"OUTPUT_DIR    : {OUTPUT_DIR}")



## 4. Run DINO EMA Pretraining
Single GPU training with plain `python`. Adjust `--epochs`, `--batch_size`, `--grad_accum` as needed.

In [ ]:
!python train_script.py \
  --model_path "$MODEL_PATH" \
  --data_path "$PUBLISHED_CSV" \
  --train_data_path "$TRAIN_CSV" \
  --output_dir "$OUTPUT_DIR" \
  --epochs 1 \
  --batch_size 2 \
  --grad_accum 16 \
  --use_bf16 true \
  --gradient_checkpointing true \
  --sample_check_every 80 \
  --save_every_steps 1000


## 5. Evaluation Utilities

In [ ]:
import sacrebleu
import pandas as pd
import os
import torch
import sys
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import numpy as np

sys.path.append('.')
from train_script import OptimizedPreprocessor

class TranslationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor: OptimizedPreprocessor):
        proc = preprocessor.preprocess_batch(df["transliteration"].tolist())
        self.sources = ["translate Akkadian to English: " + t for t in proc]
        self.references = df["translation"].tolist()

    def __len__(self):
        return len(self.sources)

    def __getitem__(self, idx):
        return self.sources[idx], self.references[idx]


# ══════════════════════════════════════════════════════════════
# Evaluation
# ══════════════════════════════════════════════════════════════

def evaluate_model(
    model_path: str,
    dataset: TranslationDataset,
    label: str,
    batch_size: int = 4,
    max_input_length: int = 512,
    max_new_tokens: int = 384,
    num_beams: int = 4,
    device: str = "cuda",
) -> dict:
    """Generate translations and compute BLEU / chrF++."""

    print(f"\n{'='*60}")
    print(f"  Evaluating: {label}")
    print(f"  Model path: {model_path}")
    print(f"{'='*60}")

    # Load model
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device).eval()
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  Parameters: {n_params:,}")

    if device == "cuda":
        used = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU mem: {used:.2f} GB")

    # Use bf16 if available
    use_bf16 = (device == "cuda" and torch.cuda.is_available()
                and getattr(torch.cuda, "is_bf16_supported", lambda: False)())

    # Generate
    all_preds = []
    all_refs = []

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    with torch.inference_mode():
        for batch_sources, batch_refs in tqdm(loader, desc=f"[{label}] Generating"):
            enc = tokenizer(
                list(batch_sources),
                max_length=max_input_length,
                padding=True,
                truncation=True,
                return_tensors="pt",
            ).to(device)

            ctx = torch.autocast("cuda", dtype=torch.bfloat16) if use_bf16 else torch.inference_mode()
            with ctx:
                outputs = model.generate(
                    input_ids=enc.input_ids,
                    attention_mask=enc.attention_mask,
                    max_new_tokens=max_new_tokens,
                    num_beams=num_beams,
                    length_penalty=1.3,
                    early_stopping=True,
                    repetition_penalty=1.2,
                    use_cache=True,
                )

            preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            all_preds.extend(preds)
            all_refs.extend(batch_refs)

    # Cleanup
    del model
    del tokenizer
    import gc
    gc.collect()
    if device == "cuda":
        torch.cuda.empty_cache()

    # Compute metrics
    bleu = sacrebleu.corpus_bleu(all_preds, [all_refs])
    chrf = sacrebleu.corpus_chrf(all_preds, [all_refs], word_order=2)  # chrF++

    results = {
        "label": label,
        "model_path": model_path,
        "n_samples": len(all_preds),
        "BLEU": bleu.score,
        "chrF++": chrf.score,
        "bleu_detail": str(bleu),
        "chrf_detail": str(chrf),
        "predictions": all_preds,
        "references": all_refs,
    }

    print(f"\n  Results for [{label}]:")
    print(f"    BLEU   : {bleu.score:.2f}")
    print(f"    chrF++ : {chrf.score:.2f}")
    print(f"    {bleu}")
    print(f"    {chrf}")

    return results


def show_comparison(res_orig: dict, res_dino: dict, n_examples: int = 10):
    """Side-by-side comparison table + sample translations."""

    print(f"\n{'='*70}")
    print(f"  COMPARISON: Original vs DINO-pretrained")
    print(f"{'='*70}")
    print(f"  {'Metric':<12} {'Original':>12} {'DINO':>12} {'Delta':>12}")
    print(f"  {'-'*48}")

    bleu_delta = res_dino["BLEU"] - res_orig["BLEU"]
    chrf_delta = res_dino["chrF++"] - res_orig["chrF++"]

    bleu_sign = "+" if bleu_delta >= 0 else ""
    chrf_sign = "+" if chrf_delta >= 0 else ""

    print(f"  {'BLEU':<12} {res_orig['BLEU']:>12.2f} {res_dino['BLEU']:>12.2f} {bleu_sign}{bleu_delta:>11.2f}")
    print(f"  {'chrF++':<12} {res_orig['chrF++']:>12.2f} {res_dino['chrF++']:>12.2f} {chrf_sign}{chrf_delta:>11.2f}")
    print(f"  {'-'*48}")

    # Per-sample chrF++ differences
    metric = sacrebleu.metrics.CHRF(word_order=2)
    orig_scores = []
    dino_scores = []
    for i in range(len(res_orig["predictions"])):
        ref = res_orig["references"][i]
        s_orig = metric.sentence_score(res_orig["predictions"][i], [ref]).score
        s_dino = metric.sentence_score(res_dino["predictions"][i], [ref]).score
        orig_scores.append(s_orig)
        dino_scores.append(s_dino)

    orig_scores = np.array(orig_scores)
    dino_scores = np.array(dino_scores)
    diffs = dino_scores - orig_scores

    n_better = (diffs > 0).sum()
    n_worse = (diffs < 0).sum()
    n_same = (diffs == 0).sum()

    print(f"\n  Per-sample chrF++ comparison (N={len(diffs)}):")
    print(f"    DINO better : {n_better} ({100*n_better/len(diffs):.1f}%)")
    print(f"    DINO worse  : {n_worse} ({100*n_worse/len(diffs):.1f}%)")
    print(f"    Same        : {n_same} ({100*n_same/len(diffs):.1f}%)")
    print(f"    Mean delta  : {diffs.mean():+.2f}")
    print(f"    Median delta: {np.median(diffs):+.2f}")

    # Show examples: biggest improvements and biggest regressions
    sorted_idx = np.argsort(diffs)

    print(f"\n  {'─'*70}")
    print(f"  Top {min(n_examples, len(diffs))} DINO improvements:")
    print(f"  {'─'*70}")
    for rank, i in enumerate(reversed(sorted_idx[-n_examples:])):
        if diffs[i] <= 0:
            break
        print(f"\n  #{rank+1} | chrF++ delta: {diffs[i]:+.1f} (orig={orig_scores[i]:.1f} → dino={dino_scores[i]:.1f})")
        print(f"  REF : {res_orig['references'][i][:120]}")
        print(f"  ORIG: {res_orig['predictions'][i][:120]}")
        print(f"  DINO: {res_dino['predictions'][i][:120]}")

    print(f"\n  {'─'*70}")
    print(f"  Top {min(n_examples, len(diffs))} DINO regressions:")
    print(f"  {'─'*70}")
    for rank, i in enumerate(sorted_idx[:n_examples]):
        if diffs[i] >= 0:
            break
        print(f"\n  #{rank+1} | chrF++ delta: {diffs[i]:+.1f} (orig={orig_scores[i]:.1f} → dino={dino_scores[i]:.1f})")
        print(f"  REF : {res_orig['references'][i][:120]}")
        print(f"  ORIG: {res_orig['predictions'][i][:120]}")
        print(f"  DINO: {res_dino['predictions'][i][:120]}")

    return {
        "n_better": int(n_better),
        "n_worse": int(n_worse),
        "n_same": int(n_same),
        "mean_delta": float(diffs.mean()),
        "median_delta": float(np.median(diffs)),
    }




## 6. Predict & Evaluate

In [ ]:
print("\n=> Preparing Evaluation Phase...")
TRAIN_CSV = os.environ.get("TRAIN_CSV", "")
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", "")
MODEL_PATH = os.environ.get("MODEL_PATH", "")

if not os.path.exists(TRAIN_CSV):
    print(f"Warning: {TRAIN_CSV} not found. Skipping evaluation.")
else:
    eval_df = pd.read_csv(TRAIN_CSV)
    eval_df = eval_df.sample(n=min(400, len(eval_df)), random_state=42).reset_index(drop=True)
    dataset = TranslationDataset(eval_df, OptimizedPreprocessor())

    device = "cuda" if torch.cuda.is_available() else "cpu"

    print("\nEvaluating Original Baseline Model...")
    try:
        res_orig = evaluate_model(MODEL_PATH, dataset, "Original", batch_size=8, max_new_tokens=256, device=device)
    except Exception as e:
        print("Failed to evaluate original model:", e)
        res_orig = None

    print("\nEvaluating DINO EMA Fine-tuned Model...")
    dino_model_path = os.path.join(OUTPUT_DIR, "final")
    try:
        res_dino = evaluate_model(dino_model_path, dataset, "DINO EMA", batch_size=8, max_new_tokens=256, device=device)
    except Exception as e:
        print("Failed to evaluate DINO model:", e)
        res_dino = None

    if res_orig is not None and res_dino is not None:
        comp = show_comparison(res_orig, res_dino, n_examples=10)



## 7. MBR Ensemble Submission
DINO-pretrained student (Model A) + mattiaangeli/byt5-akkadian-mbr (Model B)  
Cross-model candidate pooling → post-processing → chrF++ MBR → `submission.csv`

In [ ]:
%%writefile submission_script.py
#!/usr/bin/env python3
"""
MBR Ensemble Submission
=======================
Model A : DINO-pretrained student  (dino_ema_output/final/student)
Model B : mattiaangeli/byt5-akkadian-mbr
→ Cross-model candidate pooling + chrF++ MBR selection + post-processing
→ submission.csv
"""

import os
import gc
import re
import json
import math
import random
import logging
import argparse
import warnings
from pathlib import Path
from contextlib import nullcontext
from dataclasses import dataclass
from typing import List, Tuple, Dict

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, Sampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm.auto import tqdm
import sacrebleu

warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────────────────────

@dataclass
class SubmissionConfig:
    dino_model_path: str = "/kaggle/working/dino_ema_output/final/student"
    model_b_path: str    = "/kaggle/input/models/mattiaangeli/byt5-akkadian-mbr/pytorch/default/6"
    test_data_path: str  = ""   # auto-detected if empty
    lexicon_path: str    = ""   # auto-detected if empty
    output_dir: str      = "/kaggle/working"

    max_input_length: int = 512
    max_new_tokens: int   = 384
    batch_size: int       = 2
    num_workers: int      = 2
    num_buckets: int      = 6

    num_beam_cands: int       = 4
    num_beams: int            = 8
    length_penalty: float     = 1.3
    repetition_penalty: float = 1.2
    num_sample_cands: int     = 2
    mbr_top_p: float          = 0.92
    mbr_temperature: float    = 0.75
    mbr_pool_cap: int         = 32

    use_bf16: bool         = True
    use_bucket_batching: bool = True
    use_adaptive_beams: bool  = True
    checkpoint_freq: int   = 200

    def __post_init__(self):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        Path(self.output_dir).mkdir(exist_ok=True, parents=True)

        if self.device.type != "cuda":
            self.use_bf16 = False

        self.use_bf16_amp = self.use_bf16 and self._bf16_supported()

        # Auto-detect test.csv
        if not self.test_data_path:
            for root, _, files in os.walk("/kaggle/input"):
                if "test.csv" in files:
                    self.test_data_path = os.path.join(root, "test.csv")
                    break

        # Auto-detect lexicon
        if not self.lexicon_path:
            candidate = "/kaggle/input/competitions/deep-past-initiative-machine-translation/OA_Lexicon_eBL.csv"
            if os.path.exists(candidate):
                self.lexicon_path = candidate

    @staticmethod
    def _bf16_supported() -> bool:
        if not torch.cuda.is_available():
            return False
        try:
            return bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
        except Exception:
            return False


def _bf16_ctx(device: torch.device, enabled: bool):
    if enabled and device.type == "cuda":
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


# ──────────────────────────────────────────────────────────────
# Preprocessing
# ──────────────────────────────────────────────────────────────

_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz","š").replace("SZ","Š")
    s = s.replace("s,","ṣ").replace("S,","Ṣ")
    s = s.replace("t,","ṭ").replace("T,","Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s

_ALLOWED_FRACS = [
    (1/6,"0.16666"),(1/4,"0.25"),(1/3,"0.33333"),
    (1/2,"0.5"),(2/3,"0.66666"),(3/4,"0.75"),(5/6,"0.83333"),
]
_FRAC_TOL = 2e-3
_FLOAT_RE = re.compile(r"(?<![\w/])(\d+\.\d{4,})(?![\w/])")

def _canon_decimal(x: float) -> str:
    ip = int(math.floor(x + 1e-12))
    frac = x - ip
    best = min(_ALLOWED_FRACS, key=lambda t: abs(frac - t[0]))
    if abs(frac - best[0]) <= _FRAC_TOL:
        dec = best[1]
        if ip == 0:
            return dec
        return f"{ip}{dec[1:]}" if dec.startswith("0.") else f"{ip}+{dec}"
    return f"{x:.5f}".rstrip("0").rstrip(".")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>|<\s*gap\s*>|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b|\.{3,}|…+|\[\.+\]|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)", re.I
)
_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"
_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")
_WS_RE = re.compile(r"\s+")
_EXACT_FRAC_RE = re.compile(r"0\.8333|0\.6666|0\.3333|0\.1666|0\.625|0\.75|0\.25|0\.5")
_EXACT_FRAC_MAP = {"0.8333":"⅚","0.6666":"⅔","0.3333":"⅓","0.1666":"⅙","0.625":"⅝","0.75":"¾","0.25":"¼","0.5":"½"}

def _frac_repl(m: re.Match) -> str:
    return _EXACT_FRAC_MAP[m.group(0)]

class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)
        ser = ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace("ₓ", "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        ser = ser.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()


# ──────────────────────────────────────────────────────────────
# Post-processing
# ──────────────────────────────────────────────────────────────

_SOFT_GRAM_RE   = re.compile(r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)(?:\.\s*(?:plur|plural|sing|singular))?\.?\s*[^)]*\)", re.I)
_BARE_GRAM_RE   = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE   = re.compile(r"\(\?\)")
_CURLY_QUOT_RE  = re.compile("[\u201c\u201d\u2018\u2019]")
_MONTH_RE       = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT      = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}
_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE= re.compile(r"([.,])\1+")
_PUNCT_SPC_RE   = re.compile(r"\s+([.,:])")
_FORBIDDEN_TRANS= str.maketrans("","","()——<>⌈⌋⌊[]+ʾ;")
_COMMODITY_RE   = re.compile(r"-(gold|tax|textiles)\b")
_COMMODITY_REPL = {"gold":"pašallum gold","tax":"šadduātum tax","textiles":"kutānum textiles"}
_SHEKEL_REPLS   = [
    (re.compile(r"5\s+11\s*/\s*12\s+shekels?", re.I), "6 shekels less 15 grains"),
    (re.compile(r"5\s*/\s*12\s+shekels?", re.I),      "⅔ shekel 15 grains"),
    (re.compile(r"7\s*/\s*12\s+shekels?", re.I),      "½ shekel 15 grains"),
    (re.compile(r"1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?", re.I), "15 grains"),
]
_SLASH_ALT_RE   = re.compile(r"(?<!\d)\s*/\s*(?!\d)\S+")
_STRAY_MARKS_RE = re.compile(r"<<[^>]*>>|<(?!gap\b)[^>]*>")
_MULTI_GAP_RE   = re.compile(r"(?:<gap>\s*){2,}")
_PN_RE          = re.compile(r"\bPN\b")

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]

class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)
        s = s.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)
        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)
        s = s.str.replace(_EXACT_FRAC_RE, _frac_repl, regex=True)
        s = s.str.replace(_FLOAT_RE, lambda m: _canon_decimal(float(m.group(1))), regex=True)
        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)
        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        s = s.str.replace(_SLASH_ALT_RE, "", regex=True)
        s = s.str.replace(_CURLY_QUOT_RE, "", regex=True)
        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)
        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)
        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)
        s = s.str.replace(_PUNCT_SPC_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()
        return s.tolist()


# ──────────────────────────────────────────────────────────────
# Dataset + BucketBatchSampler
# ──────────────────────────────────────────────────────────────

class AkkadianDataset(Dataset):
    def __init__(self, df: pd.DataFrame, preprocessor: OptimizedPreprocessor):
        self.sample_ids = df["id"].tolist()
        proc = preprocessor.preprocess_batch(df["transliteration"].tolist())
        self.input_texts = ["translate Akkadian to English: " + t for t in proc]

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        return self.sample_ids[idx], self.input_texts[idx]


class BucketBatchSampler(Sampler):
    def __init__(self, dataset: AkkadianDataset, batch_size: int, num_buckets: int):
        lengths = [len(t.split()) for _, t in dataset]
        sorted_idx = sorted(range(len(lengths)), key=lambda i: lengths[i])
        bsize = max(1, len(sorted_idx) // max(1, num_buckets))
        self.buckets = [
            sorted_idx[i * bsize: None if i == num_buckets - 1 else (i + 1) * bsize]
            for i in range(num_buckets)
        ]
        self.batch_size = batch_size

    def __iter__(self):
        for bucket in self.buckets:
            for i in range(0, len(bucket), self.batch_size):
                yield bucket[i:i + self.batch_size]

    def __len__(self):
        return sum(math.ceil(len(b) / self.batch_size) for b in self.buckets)


# ──────────────────────────────────────────────────────────────
# MBR Selector
# ──────────────────────────────────────────────────────────────

def _load_lexicon(lexicon_path: str) -> Dict[str, str]:
    if not lexicon_path or not os.path.exists(lexicon_path):
        return {}
    df = pd.read_csv(lexicon_path, encoding="utf-8")
    target_types = ["PN", "GN", "DN", "RN"]
    entity_df = df[df["type"].isin(target_types)].copy()
    lexicon = {}
    for _, row in entity_df.iterrows():
        form = str(row["form"]).strip()
        norm = str(row["norm"]).strip()
        if form == "nan" or norm == "nan":
            continue
        clean = re.sub(r"[\[\]\(\)\?\!]", "", form).lower()
        if clean:
            lexicon[clean] = norm
    print(f"Loaded {len(lexicon)} proper nouns into lexicon.")
    return lexicon


class MBRSelector:
    def __init__(self, pool_cap: int = 32, lexicon: Dict[str, str] = None):
        self._metric = sacrebleu.metrics.CHRF(word_order=2)
        self.pool_cap = pool_cap
        self.lexicon = lexicon or {}
        self.w_chrf = 0.8 if self.lexicon else 1.0
        self.w_fidelity = 0.2 if self.lexicon else 0.0

    def _chrfpp(self, a: str, b: str) -> float:
        a, b = (a or "").strip(), (b or "").strip()
        if not a or not b:
            return 0.0
        return float(self._metric.sentence_score(a, [b]).score)

    def _fidelity(self, source: str, candidate: str) -> float:
        if not self.lexicon or not source or not candidate:
            return 100.0
        tokens = re.sub(r"[^\w\-\s]", "", source.lower()).split()
        expected = [self.lexicon[t].lower() for t in tokens if t in self.lexicon]
        if not expected:
            return 100.0
        cand_lower = candidate.lower()
        return (sum(1 for e in expected if e in cand_lower) / len(expected)) * 100.0

    @staticmethod
    def _dedup(xs: List[str]) -> List[str]:
        seen, out = set(), []
        for x in xs:
            x = str(x).strip()
            if x and x not in seen:
                out.append(x)
                seen.add(x)
        return out

    def pick(self, source: str, candidates: List[str]) -> str:
        cands = self._dedup(candidates)[:self.pool_cap]
        n = len(cands)
        if n == 0:
            return ""
        if n == 1:
            return cands[0]

        best_i, best_s = 0, -1e9
        for i in range(n):
            consensus = sum(self._chrfpp(cands[i], cands[j]) for j in range(n) if j != i) / max(1, n - 1)
            fidelity = self._fidelity(source, cands[i])
            score = self.w_chrf * consensus + self.w_fidelity * fidelity
            if score > best_s:
                best_s, best_i = score, i
        return cands[best_i]


# ──────────────────────────────────────────────────────────────
# Model Wrapper
# ──────────────────────────────────────────────────────────────

class ModelWrapper:
    def __init__(self, model_path: str, cfg: SubmissionConfig, label: str):
        self.cfg = cfg
        self.label = label
        print(f"[{label}] Loading from {model_path}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(cfg.device).eval()
        n = sum(p.numel() for p in self.model.parameters())
        print(f"[{label}] {n:,} parameters")

    def collate(self, batch_samples):
        ids   = [s[0] for s in batch_samples]
        texts = [s[1] for s in batch_samples]
        enc = self.tokenizer(
            texts,
            max_length=self.cfg.max_input_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        return ids, enc

    def generate_candidates(self, input_ids, attention_mask) -> List[List[str]]:
        cfg = self.cfg
        B = input_ids.shape[0]
        ctx = _bf16_ctx(cfg.device, cfg.use_bf16_amp)

        if cfg.use_adaptive_beams:
            med = float(attention_mask.sum(dim=1).float().median().item())
            beam_size = cfg.num_beams if med >= 100 else max(cfg.num_beam_cands, cfg.num_beams // 2)
        else:
            beam_size = cfg.num_beams

        with ctx:
            beam_out = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                do_sample=False,
                num_beams=beam_size,
                num_return_sequences=cfg.num_beam_cands,
                max_new_tokens=cfg.max_new_tokens,
                length_penalty=cfg.length_penalty,
                early_stopping=True,
                repetition_penalty=cfg.repetition_penalty,
                use_cache=True,
            )
            beam_texts = self.tokenizer.batch_decode(beam_out, skip_special_tokens=True)

            samp_texts = []
            if cfg.num_sample_cands > 0:
                samp_out = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    do_sample=True,
                    num_beams=1,
                    top_p=cfg.mbr_top_p,
                    temperature=cfg.mbr_temperature,
                    num_return_sequences=cfg.num_sample_cands,
                    max_new_tokens=cfg.max_new_tokens,
                    repetition_penalty=cfg.repetition_penalty,
                    use_cache=True,
                )
                samp_texts = self.tokenizer.batch_decode(samp_out, skip_special_tokens=True)

        Rb, Rs = cfg.num_beam_cands, cfg.num_sample_cands
        pools = []
        for i in range(B):
            p = list(beam_texts[i * Rb:(i + 1) * Rb])
            if Rs > 0:
                p += list(samp_texts[i * Rs:(i + 1) * Rs])
            pools.append(p)
        return pools

    def unload(self):
        del self.model
        del self.tokenizer
        self.model = None
        self.tokenizer = None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.synchronize()
        print(f"[{self.label}] Unloaded.")


# ──────────────────────────────────────────────────────────────
# Inference + MBR
# ──────────────────────────────────────────────────────────────

def run_model(wrapper: ModelWrapper, dataset: AkkadianDataset, cfg: SubmissionConfig) -> Dict[str, List[str]]:
    if cfg.use_bucket_batching:
        sampler = BucketBatchSampler(dataset, cfg.batch_size, cfg.num_buckets)
        dl = DataLoader(
            dataset,
            batch_sampler=sampler,
            num_workers=cfg.num_workers,
            collate_fn=wrapper.collate,
            pin_memory=(cfg.device.type == "cuda"),
        )
    else:
        dl = DataLoader(
            dataset,
            batch_size=cfg.batch_size,
            shuffle=False,
            num_workers=cfg.num_workers,
            collate_fn=wrapper.collate,
            pin_memory=(cfg.device.type == "cuda"),
        )

    pools_by_id = {}
    with torch.inference_mode():
        for batch_ids, enc in tqdm(dl, desc=f"[{wrapper.label}]"):
            input_ids = enc.input_ids.to(cfg.device, non_blocking=True)
            attn = enc.attention_mask.to(cfg.device, non_blocking=True)
            try:
                batch_pools = wrapper.generate_candidates(input_ids, attn)
                for sid, pool in zip(batch_ids, batch_pools):
                    pools_by_id[str(sid)] = pool
            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(f"[{wrapper.label}] OOM — skipping batch")
                    torch.cuda.empty_cache()
                    for sid in batch_ids:
                        pools_by_id.setdefault(str(sid), [])
                else:
                    raise
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    return pools_by_id


def run_submission(cfg: SubmissionConfig):
    # Logging
    logging.basicConfig(
        level=logging.INFO,
        format="%(asctime)s | %(message)s",
        handlers=[
            logging.StreamHandler(),
            logging.FileHandler(Path(cfg.output_dir) / "submission_mbr.log"),
        ],
    )
    logger = logging.getLogger("submission_mbr")

    logger.info("=" * 60)
    logger.info("MBR Ensemble Submission")
    logger.info(f"  Model A (DINO) : {cfg.dino_model_path}")
    logger.info(f"  Model B        : {cfg.model_b_path}")
    logger.info(f"  Test data      : {cfg.test_data_path}")
    logger.info(f"  BF16 AMP       : {cfg.use_bf16_amp}")
    logger.info("=" * 60)

    if not cfg.test_data_path or not os.path.exists(cfg.test_data_path):
        raise FileNotFoundError(f"test.csv not found: '{cfg.test_data_path}'")

    test_df = pd.read_csv(cfg.test_data_path, encoding="utf-8")
    logger.info(f"Test samples: {len(test_df)}")

    preprocessor = OptimizedPreprocessor()
    postprocessor = VectorizedPostprocessor()
    lexicon = _load_lexicon(cfg.lexicon_path)
    mbr = MBRSelector(pool_cap=cfg.mbr_pool_cap, lexicon=lexicon)
    dataset = AkkadianDataset(test_df, preprocessor)

    # Phase 1: Model A (DINO student)
    logger.info("Phase 1/2 — Model A (DINO student)")
    wrapper_a = ModelWrapper(cfg.dino_model_path, cfg, "DINO-student")
    pools_a = run_model(wrapper_a, dataset, cfg)
    wrapper_a.unload()
    del wrapper_a

    # Phase 2: Model B
    logger.info("Phase 2/2 — Model B")
    wrapper_b = ModelWrapper(cfg.model_b_path, cfg, "Model-B")
    pools_b = run_model(wrapper_b, dataset, cfg)
    wrapper_b.unload()
    del wrapper_b

    # Phase 3: Pool merge + MBR + postprocessing
    logger.info("Phase 3/3 — Pool merge + MBR selection")
    results: List[Tuple[str, str]] = []

    for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="MBR"):
        sid = str(row["id"])
        source = str(row["transliteration"])

        combined = pools_a.get(sid, []) + pools_b.get(sid, [])
        pp = postprocessor.postprocess_batch(combined) if combined else []
        chosen = mbr.pick(source, pp)

        if not chosen or not chosen.strip():
            chosen = "The tablet is too damaged to translate."

        results.append((sid, chosen))

        if len(results) % cfg.checkpoint_freq == 0:
            ckpt = Path(cfg.output_dir) / f"submission_ckpt_{len(results)}.csv"
            pd.DataFrame(results, columns=["id", "translation"]).to_csv(ckpt, index=False)
            logger.info(f"  Checkpoint: {len(results)} rows → {ckpt}")

    result_df = pd.DataFrame(results, columns=["id", "translation"])

    # Stats
    empty = result_df["translation"].str.strip().eq("").sum()
    lens = result_df["translation"].str.len()
    logger.info(f"Empty: {empty} | Len mean: {lens.mean():.1f} median: {lens.median():.1f}")

    out_path = Path(cfg.output_dir) / "submission.csv"
    result_df.to_csv(out_path, index=False)
    logger.info(f"Saved → {out_path} ({len(result_df)} rows)")
    print(f"\nSubmission saved: {out_path}")
    return result_df


# ──────────────────────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────────────────────

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--dino_model_path", type=str, default="")
    parser.add_argument("--model_b_path",    type=str, default="")
    parser.add_argument("--test_data_path",  type=str, default="")
    parser.add_argument("--output_dir",      type=str, default="/kaggle/working")
    parser.add_argument("--batch_size",      type=int, default=2)
    parser.add_argument("--num_beams",       type=int, default=8)
    parser.add_argument("--num_beam_cands",  type=int, default=4)
    parser.add_argument("--num_sample_cands",type=int, default=2)
    parser.add_argument("--max_new_tokens",  type=int, default=384)
    parser.add_argument("--mbr_pool_cap",    type=int, default=32)
    parser.add_argument("--use_bf16",        type=lambda x: x.lower() in ("true","1"), default=True)
    args = parser.parse_args()

    cfg = SubmissionConfig(**{k: v for k, v in vars(args).items() if v})
    run_submission(cfg)


In [ ]:
!python submission_script.py \
  --dino_model_path "$OUTPUT_DIR/final/student" \
  --output_dir "$OUTPUT_DIR" \
  --batch_size 2 \
  --num_beams 8 \
  --max_new_tokens 384
